In [ ]:
# file: shor_9qubit_qiskit.py

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from math import gcd
from fractions import Fraction
import numpy as np

# Number to factor
N = 15
a = 2  # Random co-prime with N

# --- Helper Functions ---

def c_amod15(a, power):
    """Controlled modular multiplication by a^power mod 15."""
    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 7, 8, 13]:
            U.swap(0, 1)
            U.swap(1, 2)
            U.swap(2, 3)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)
    U = U.to_gate()
    U.name = f"{a}^{power} mod 15"
    cU = U.control()
    return cU


def qft_dagger(n):
    """Inverse Quantum Fourier Transform."""
    qc = QuantumCircuit(n)
    for qubit in range(n // 2):
        qc.swap(qubit, n - qubit - 1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / float(2 ** (j - m)), m, j)
        qc.h(j)
    qc.name = "QFT†"
    return qc


# --- Build Circuit ---

n_count = 4  # Counting qubits
qc = QuantumCircuit(4 + 4, 4)

# Step 1: Apply Hadamard to counting qubits
qc.h(range(4))

# Step 2: Initialize work register to |1>
qc.x(4 + 3)

# Step 3: Controlled modular exponentiation
for q in range(n_count):
    qc.append(c_amod15(a, 2 ** q), [q] + [i + 4 for i in range(4)])

# Step 4: Apply inverse QFT
qc.append(qft_dagger(n_count), range(n_count))

# Step 5: Measure counting register
qc.measure(range(n_count), range(n_count))

# --- Simulation ---

sim = AerSimulator.
qc = transpile(qc, sim)
result = execute(qc, sim, shots=4096).result()
counts = result.get_counts()

# --- Extract Results ---

plot_histogram(counts)

# Get measured phase
measured = max(counts, key=counts.get)
phase = int(measured, 2) / 2**n_count
frac = Fraction(phase).limit_denominator(N)
r = frac.denominator
print(f"Measured phase = {phase}, possible r = {r}")

# --- Classical Post-processing ---
if r % 2 == 0:
    factor1 = gcd(a ** (r // 2) - 1, N)
    factor2 = gcd(a ** (r // 2) + 1, N)
    if factor1 not in [1, N]:
        print(f"Factors of {N}: {factor1} and {factor2}")
    else:
        print("Failed to find non-trivial factors.")
else:
    print("r is odd; retry with different 'a'.")


NameError: name 'Aer' is not defined